# 뉴스 검증하기 위해서 사용하는 코드
과거 뉴스 스크랩 한 뒤 상승 하락 칼럼 만들고 뉴스 본문 요약 후 쪼개서 감정분석

In [ ]:
import pandas as pd
dfdf = pd.read_csv("/Users/kjb/Desktop/hateslop/프로젝트/newtest.csv")
df = dfdf.copy()
df = df.iloc[:1000]

In [43]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import pandas as pd

# ✅ 모델 로딩
model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, use_safetensors=True)
summarizer = pipeline("summarization", model=model, tokenizer=tokenizer)

# ✅ 긴 본문 자동 분할 요약 함수
def summarize_text(text, max_chunk_tokens=900):
    inputs = tokenizer(text, return_tensors="pt", truncation=False)
    input_ids = inputs["input_ids"][0]
    total_tokens = len(input_ids)

    if total_tokens <= max_chunk_tokens:
        return summarizer(text, max_length=100, min_length=20, do_sample=False)[0]["summary_text"]

    summaries = []
    for i in range(0, total_tokens, max_chunk_tokens):
        chunk_ids = input_ids[i:i + max_chunk_tokens]
        chunk_text = tokenizer.decode(chunk_ids, skip_special_tokens=True)
        summary = summarizer(chunk_text, max_length=80, min_length=15, do_sample=False)[0]["summary_text"]
        summaries.append(summary)

    return " ".join(summaries)

# ✅ DataFrame 불러오기 및 요약 칼럼 추가
# df = pd.read_csv("your_file.csv")  # 필요 시 주석 해제
summaries = []

for i, text in enumerate(df["본문"]):
    try:
        summary = summarize_text(text)
    except Exception as e:
        print(f"[{i}번째 요약 실패] {e}")
        summary = "[요약 실패]"
    summaries.append(summary)

df["요약"] = summaries



Device set to use mps:0
Your max_length is set to 80, but your input_length is only 73. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=36)
Your max_length is set to 80, but your input_length is only 39. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=19)
Your max_length is set to 80, but your input_length is only 31. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=15)
Your max_length is set to 100, but your input_length is only 80. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...'

# 중립 제거 이거 안쓸듯

In [ ]:
import pandas as pd
import re
import spacy
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

df1 = df.copy()
# 2. spaCy 로드
nlp = spacy.load("en_core_web_sm")

# 3. NER 파이프라인
from transformers import pipeline as hf_pipeline
ner = hf_pipeline("ner", model="dslim/bert-base-NER", tokenizer="dslim/bert-base-NER", aggregation_strategy="simple")

# 4. 감성 분석 파이프라인
model_name = "yiyanghkust/finbert-tone"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, trust_remote_code=True, use_safetensors=True)
sentiment = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, return_all_scores=True)

# 5. 결과 저장용 리스트
sentiment_results = []

# 6. 한 줄씩 처리
for _, row in df1.iterrows():
    text = row["요약"]
    company_name = row["회사"]

    # 👉 본문을 접속사 기준으로 분리
    split_points = [m.start() for m in re.finditer(r'\bwhile\b|\bbut\b', text)]
    clauses = []
    start = 0
    for point in split_points:
        clauses.append(text[start:point].strip())
        start = point
    clauses.append(text[start:].strip())

    # 👉 동사 맨 앞에 나오면 이전 clause와 합치기
    final_clauses = []
    for i, clause in enumerate(clauses):
        doc = nlp(clause)
        if i > 0 and len(doc) > 0 and doc[0].pos_ == "VERB":
            final_clauses[-1] += " " + clause
        else:
            final_clauses.append(clause)

    found = False
    for clause in final_clauses:
        ents = ner(clause)
        orgs = [e["word"] for e in ents if e["entity_group"] == "ORG"]

        if not orgs and company_name in clause:
            orgs.append(company_name)

        if any(company_name in o for o in orgs):
            preds = sentiment(clause)[0]  # 2중 리스트에서 첫 번째 결과 꺼냄 (list of dicts)
            top = max(preds, key=lambda x: x["score"])  # 가장 높은 점수 선택

            if top["label"] == "Neutral":
                filtered = [p for p in preds if p["label"] in ["Positive", "Negative"]]
                if filtered:
                    top = max(filtered, key=lambda x: x["score"])

            sentiment_results.append(top["label"])
            found = True
            break

    if not found:
        preds = sentiment(text[:512])[0]  # ← 이 줄이 반드시 있어야 함!
        top = max(preds, key=lambda x: x["score"])

        if top["label"] == "Neutral":
            filtered = [p for p in preds if p["label"] in ["Positive", "Negative"]]
            if filtered:
                top = max(filtered, key=lambda x: x["score"])

        sentiment_results.append(top["label"])

# 7. 결과 저장
df1["감성분석"] = sentiment_results


Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0
Device set to use mps:0
/opt/anaconda3/envs/hateslop1/lib/python3.10/site-packages/transformers/pipelines/text_classification.py:106: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores

In [55]:
import pandas as pd
import re
import spacy
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

df1 = df.copy()
# 2. spaCy 로드
nlp = spacy.load("en_core_web_sm")

# 3. NER 파이프라인
from transformers import pipeline as hf_pipeline
ner = hf_pipeline("ner", model="dslim/bert-base-NER", tokenizer="dslim/bert-base-NER", aggregation_strategy="simple")

# 4. 감성 분석 파이프라인
model_name = "yiyanghkust/finbert-tone"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, trust_remote_code=True, use_safetensors=True)
sentiment = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, return_all_scores=True)

# 5. 결과 저장용 리스트
sentiment_results = []
for _, row in df1.iterrows():
    try:
        text = row["요약"]
        company_name = row["회사"]

        # 1. 문장 분리
        split_points = [m.start() for m in re.finditer(r'\bwhile\b|\bbut\b', text)]
        clauses = []
        start = 0
        for point in split_points:
            clauses.append(text[start:point].strip())
            start = point
        clauses.append(text[start:].strip())

        # 2. 병합
        final_clauses = []
        for i, clause in enumerate(clauses):
            doc = nlp(clause)
            if i > 0 and len(doc) > 0 and doc[0].pos_ == "VERB":
                final_clauses[-1] += " " + clause
            else:
                final_clauses.append(clause)

        found = False
        for clause in final_clauses:
            ents = ner(clause)
            orgs = [e["word"] for e in ents if e["entity_group"] == "ORG"]

            if not orgs and company_name in clause:
                orgs.append(company_name)

            if any(company_name in o for o in orgs):
                preds = sentiment(clause)[0]
                top = max(preds, key=lambda x: x["score"])
                sentiment_results.append(top["label"])
                found = True
                break

        if not found:
            preds = sentiment(text[:512])[0]
            top = max(preds, key=lambda x: x["score"])
            sentiment_results.append(top["label"])

    except Exception as e:
        print(f"❌ 예외 발생: {e}")
        sentiment_results.append("Unknown")
df1["감성분석"] = sentiment_results

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0
Device set to use mps:0


In [ ]:
# 조건에 맞는 경우의 수 계산
match_positive = df2[(df["상승여부"] == 1.0) & (df2["감성분석"] == "Positive")].shape[0]
match_negative = df2[(df["상승여부"] == 0.0) & (df2["감성분석"] == "Negative")].shape[0]
total = df2[(df2["감성분석"] == "Positive") | (df2["감성분석"] == "Negative")].shape[0]
predict_positive = df2[df2["감성분석"] == "Positive"].shape[0]
real_positive = df2[df2["상승여부"] == 1.0].shape[0]
# 전체 정확도 계산
correct = match_positive + match_negative
accuracy = correct / total
positive_accuracy = real_positive / predict_positive
print(f"🔼 상승=1 & 긍정 일치: {match_positive}건")
print(f"🔽 하락=0 & 부정 일치: {match_negative}건")
print(f"✅ 총 일치: {correct} / {total} (정확도: {accuracy:.2%})")
print(f"✅ 총 일치: {real_positive} / {predict_positive} (정확도: {positive_accuracy:.2%})")

🔼 상승=1 & 긍정 일치: 88건
🔽 하락=0 & 부정 일치: 40건
✅ 총 일치: 128 / 300 (정확도: 42.67%)
✅ 총 일치: 144 / 169 (정확도: 85.21%)
